In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import pandas as pd
from math import comb

def binom_p(k, n):  # exact two-sided binomial, p=0.5
    if n == 0: return 1.0
    return min(1.0, 2 * sum(comb(n,i) for i in range(0, min(k,n-k)+1)) / 2**n)

wf = pd.read_csv("Ablation_5seed_results.csv")
nf = pd.read_csv("Ablation_5seed_NoFuzzy_results.csv")

def test(a, b, label):  # a, b: bool Series indexed by (Seed, Question ID)
    a, b = a.align(b, join='inner')
    b01, b10 = int((~a & b).sum()), int((a & ~b).sum())
    d = b.groupby('Question ID').mean() - a.groupby('Question ID').mean()
    nz = d[d != 0]; pos, neg = int((nz>0).sum()), int((nz<0).sum())
    print(f"{label}: pooled {b01}/{b10} p={binom_p(min(b01,b10), b01+b10):.5f} | "
          f"q-level n={len(nz)} (+{pos}/-{neg}) sign p={binom_p(min(pos,neg), pos+neg):.5f} | "
          f"diff={d.mean()*100:+.2f}pp")

def col(df, c): return df.set_index(['Seed','Question ID'])[c].astype(bool)

test(col(wf,'Fuzzy Correct'), col(wf,'Self Correction Fuzzy Correct'), 'SC | fuzzy ON ')
test(col(nf,'Raw Correct'),   col(nf,'Self Correction Correct'),       'SC | fuzzy OFF')
test(col(nf,'Raw Correct'),   col(wf,'Fuzzy Correct'),                 'Fuzzy | SC OFF')
test(col(nf,'Self Correction Correct'), col(wf,'Self Correction Fuzzy Correct'), 'Fuzzy | SC ON ')
test(col(nf,'Raw Correct'),   col(wf,'Self Correction Fuzzy Correct'), 'JOINT none-full')